# Sanskrit → English NMT — **Training Notebook**

**Course:** Natural Language Understanding — Assignment 2
**Role of this file:** trains the translation model on the provided data and saves everything an
evaluator needs (`evaluate.ipynb` reloads those artifacts to score the private test set).

**Architecture:** a custom Transformer encoder–decoder built from scratch in PyTorch — attention,
masking, beam search all written out. **No pre-trained translation weights.** The only pre-trained
model anywhere is the one BERTScore loads internally to score us (disclosed at the bottom).

### Data
Uses only the six provided files (`train_sa_10000.csv`, `train_en_10000.csv`,
`dev_sa_1000.csv`, `dev_en_1000.csv`, `test_sa_1000.csv`, `test_en_1000.csv`). Set `DATA_DIR`
below to wherever they live (on Colab, upload them or mount Drive).

### Artifacts this notebook writes (commit these to GitHub)
`artifacts/best_model.pt`, `artifacts/spm_sa.model`, `artifacts/spm_en.model`,
`artifacts/config.json`, `artifacts/training_curves.png`, and `submission.csv`.

## 0. Install dependencies

In [ ]:
%pip install -q torch sentencepiece nltk bert-score pandas matplotlib tqdm

## 1. Imports, configuration, reproducibility

Every knob lives in `CFG`. Device order is CUDA → Apple-Silicon MPS → CPU, so the same notebook
runs on Colab or a Mac. **Set `DATA_DIR`** to the folder holding the six CSVs.

In [ ]:
import os, math, time, json, random, unicodedata
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

def pick_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

@dataclass
class CFG:
    data_dir: str = "."          # <-- folder with the six provided CSVs
    work_dir: str = "artifacts"  # where checkpoints + tokenizers are saved

    src_vocab: int = 8000
    tgt_vocab: int = 8000
    pad_id: int = 0
    unk_id: int = 1
    bos_id: int = 2
    eos_id: int = 3

    d_model: int = 256
    num_heads: int = 8
    num_layers: int = 4
    d_ff: int = 1024
    dropout: float = 0.1
    max_len: int = 160           # a few provided sentences are long; 160 covers the vast majority

    batch_size: int = 64
    epochs: int = 25
    warmup_steps: int = 2000
    label_smoothing: float = 0.1
    grad_clip: float = 1.0
    patience: int = 5
    seed: int = 42

    beam_size: int = 5
    length_penalty: float = 0.6

cfg = CFG()
os.makedirs(cfg.work_dir, exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

set_seed(cfg.seed)
DEVICE = pick_device()
print("Device:", DEVICE, "| data dir:", os.path.abspath(cfg.data_dir))

## 2. Load and align the data

The Sanskrit and English sides live in separate files linked by `Source_id`. We locate each file
(the helper tolerates the `_10000`/`_1000` suffixes), merge on the id, and keep only complete pairs.
Cleaning is deliberately light — NFC-normalize, collapse whitespace, strip — so Sanskrit signal
(sandhi, casing-free Devanagari) is preserved.

In [ ]:
def find_csv(data_dir, split, side):
    """Locate a CSV for a given split ('train'/'dev'/'test') and side ('sa'/'en'),
    tolerant of suffixes like _10000 / _1000 in the provided filenames."""
    import glob
    hits = []
    for f in glob.glob(os.path.join(data_dir, "*.csv")):
        name = os.path.basename(f).lower()
        side_ok = (f"_{side}_" in name) or (f"_{side}." in name) or name.endswith(f"{side}.csv")
        if split in name and side_ok:
            hits.append(f)
    if not hits:
        raise FileNotFoundError(f"No CSV found for split='{split}', side='{side}' in {data_dir}")
    return sorted(hits)[0]


def _find_col(df, *keywords):
    for c in df.columns:
        name = c.strip().lower().replace(" ", "_")
        if all(k in name for k in keywords):
            return c
    raise KeyError(f"No column matching {keywords} in {list(df.columns)}")


def clean(text):
    text = unicodedata.normalize("NFC", str(text))
    text = " ".join(text.split())      # collapse runs of whitespace/newlines/tabs
    return text.strip()


def build_pairs(split, need_target=True):
    sa = pd.read_csv(find_csv(cfg.data_dir, split, "sa"))
    sa = pd.DataFrame({"id": sa[_find_col(sa, "id")],
                       "src": sa[_find_col(sa, "sentence", "sa")].map(clean)})
    if need_target:
        en = pd.read_csv(find_csv(cfg.data_dir, split, "en"))
        en = pd.DataFrame({"id": en[_find_col(en, "id")],
                           "tgt": en[_find_col(en, "sentence", "en")].map(clean)})
        return sa.merge(en, on="id", how="inner").dropna(subset=["src", "tgt"]).reset_index(drop=True)
    sa["tgt"] = None
    return sa

train_df = build_pairs("train")
dev_df   = build_pairs("dev")
test_df  = build_pairs("test")          # provided test ships with references
print(f"train {len(train_df)} | dev {len(dev_df)} | test {len(test_df)}")
train_df.head(3)

### Sequence-length sanity check
Informs `max_len`. If the 99th percentile is well under 160 you can lower `cfg.max_len` for speed.

In [ ]:
for name, df in [("src", train_df["src"]), ("tgt", train_df["tgt"])]:
    L = df.str.split().map(len)
    print(f"{name}: median {int(L.median())}, 95th {int(L.quantile(.95))}, "
          f"99th {int(L.quantile(.99))}, max {int(L.max())}")

## 3. Subword tokenization (SentencePiece)

Word-level vocabularies drown in Sanskrit's inflected/compound forms, so we learn subwords. Two
separate **unigram** models (the scripts don't overlap, so a shared vocab would waste slots).
Special tokens are pinned: `pad=0, unk=1, bos=2, eos=3`. The trained `.model` files are saved to
`artifacts/` and reloaded verbatim by the evaluation notebook.

In [ ]:
import sentencepiece as spm

def train_spm(sentences, model_prefix, vocab_size):
    txt_path = model_prefix + "_corpus.txt"
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write("\n".join(sentences))
    spm.SentencePieceTrainer.train(
        input=txt_path, model_prefix=model_prefix, vocab_size=vocab_size,
        model_type="unigram", character_coverage=1.0,
        pad_id=cfg.pad_id, unk_id=cfg.unk_id, bos_id=cfg.bos_id, eos_id=cfg.eos_id,
        pad_piece="<pad>", unk_piece="<unk>", bos_piece="<s>", eos_piece="</s>",
    )
    sp = spm.SentencePieceProcessor(); sp.load(model_prefix + ".model")
    return sp

# If you hit "vocab_size too high", lower cfg.src_vocab / cfg.tgt_vocab.
sp_src = train_spm(train_df["src"].tolist(), f"{cfg.work_dir}/spm_sa", cfg.src_vocab)
sp_tgt = train_spm(train_df["tgt"].tolist(), f"{cfg.work_dir}/spm_en", cfg.tgt_vocab)
cfg.src_vocab = sp_src.get_piece_size()
cfg.tgt_vocab = sp_tgt.get_piece_size()
print("vocab — SA:", cfg.src_vocab, "| EN:", cfg.tgt_vocab)
print("sample SA pieces:", sp_src.encode(train_df["src"].iloc[1], out_type=str)[:12])

## 4. Dataset and batching

Source → subword ids. Target → `[<s>] ... [</s>]`. The collate functions pad each batch to its own
max length; over-long pairs are truncated to `max_len`.

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, df, sp_src, sp_tgt, max_len, has_target=True):
        self.rows = []
        for _, r in df.iterrows():
            src_ids = sp_src.encode(r["src"], out_type=int)[: max_len - 2]
            if has_target and r["tgt"] is not None:
                tgt_ids = sp_tgt.encode(r["tgt"], out_type=int)[: max_len - 2]
                tgt_ids = [cfg.bos_id] + tgt_ids + [cfg.eos_id]
            else:
                tgt_ids = None
            self.rows.append((r["id"], src_ids, tgt_ids))

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        return self.rows[i]


def pad_batch(seqs, pad_id):
    m = max(len(s) for s in seqs)
    return torch.tensor([s + [pad_id] * (m - len(s)) for s in seqs], dtype=torch.long)


def collate_train(batch):
    _, src, tgt = zip(*batch)
    return pad_batch(list(src), cfg.pad_id), pad_batch(list(tgt), cfg.pad_id)


def collate_infer(batch):
    ids, src, _ = zip(*batch)
    return list(ids), pad_batch(list(src), cfg.pad_id)

train_ds = TranslationDataset(train_df, sp_src, sp_tgt, cfg.max_len)
dev_ds   = TranslationDataset(dev_df,   sp_src, sp_tgt, cfg.max_len)
test_ds  = TranslationDataset(test_df,  sp_src, sp_tgt, cfg.max_len)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,  collate_fn=collate_train)
dev_loader   = DataLoader(dev_ds,   batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_train)
print("train batches/epoch:", len(train_loader))

## 5. The model — a Transformer, built by hand

A standard encoder–decoder Transformer, spelled out component by component. Design choices for this
small, low-resource setting: **pre-norm** residuals (more stable at this scale), **weight tying**
between the target embedding and output projection (fewer parameters → better efficiency score),
and fixed **sinusoidal** positions. A single boolean mask handles both padding and the decoder's
causal "no peeking ahead" constraint.

In [ ]:
class PositionalEncoding(nn.Module):
    """Adds a fixed sinusoidal signal so the model can tell positions apart."""
    def __init__(self, d_model, max_len, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, : x.size(1)])


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads
        self.h = num_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        B = query.size(0)
        split = lambda x: x.view(B, -1, self.h, self.d_k).transpose(1, 2)
        q, k, v = split(self.q_proj(query)), split(self.k_proj(key)), split(self.v_proj(value))

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))
        attn = self.dropout(torch.softmax(scores, dim=-1))
        ctx = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, -1, self.h * self.d_k)
        return self.out_proj(ctx)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(d_ff, d_model),
        )
    def forward(self, x):
        return self.net(x)


class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask):
        h = self.norm1(x)
        x = x + self.dropout(self.self_attn(h, h, h, src_mask))
        h = self.norm2(x)
        x = x + self.dropout(self.ff(h))
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn  = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, memory, tgt_mask, src_mask):
        h = self.norm1(x)
        x = x + self.dropout(self.self_attn(h, h, h, tgt_mask))
        h = self.norm2(x)
        x = x + self.dropout(self.cross_attn(h, memory, memory, src_mask))
        h = self.norm3(x)
        x = x + self.dropout(self.ff(h))
        return x


class TransformerNMT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        d = cfg.d_model
        self.src_embed = nn.Embedding(cfg.src_vocab, d, padding_idx=cfg.pad_id)
        self.tgt_embed = nn.Embedding(cfg.tgt_vocab, d, padding_idx=cfg.pad_id)
        self.pos = PositionalEncoding(d, cfg.max_len, cfg.dropout)
        self.encoder = nn.ModuleList(
            [EncoderLayer(d, cfg.num_heads, cfg.d_ff, cfg.dropout) for _ in range(cfg.num_layers)])
        self.decoder = nn.ModuleList(
            [DecoderLayer(d, cfg.num_heads, cfg.d_ff, cfg.dropout) for _ in range(cfg.num_layers)])
        self.enc_norm = nn.LayerNorm(d)
        self.dec_norm = nn.LayerNorm(d)
        self.generator = nn.Linear(d, cfg.tgt_vocab)
        self.generator.weight = self.tgt_embed.weight   # weight tying
        self._init()

    def _init(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def make_src_mask(self, src):
        return (src == self.cfg.pad_id).unsqueeze(1).unsqueeze(2)

    def make_tgt_mask(self, tgt):
        T = tgt.size(1)
        pad = (tgt == self.cfg.pad_id).unsqueeze(1).unsqueeze(2)
        causal = torch.triu(torch.ones(T, T, device=tgt.device), 1).bool()
        return pad | causal.unsqueeze(0).unsqueeze(1)

    def encode(self, src, src_mask):
        x = self.pos(self.src_embed(src) * math.sqrt(self.cfg.d_model))
        for layer in self.encoder:
            x = layer(x, src_mask)
        return self.enc_norm(x)

    def decode(self, tgt, memory, tgt_mask, src_mask):
        x = self.pos(self.tgt_embed(tgt) * math.sqrt(self.cfg.d_model))
        for layer in self.decoder:
            x = layer(x, memory, tgt_mask, src_mask)
        return self.dec_norm(x)

    def forward(self, src, tgt):
        src_mask = self.make_src_mask(src)
        tgt_mask = self.make_tgt_mask(tgt)
        memory = self.encode(src, src_mask)
        out = self.decode(tgt, memory, tgt_mask, src_mask)
        return self.generator(out)

model = TransformerNMT(cfg).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model ready — {n_params:,} parameters")

## 6. Loss and optimization

**Label smoothing (0.1)** keeps the model from over-committing to a single token and reliably helps
BLEU. **Noam warmup** ramps the learning rate up for the first few thousand steps, then decays as
`1/√step` — the piece that keeps Transformer training from diverging early.

In [ ]:
class LabelSmoothingLoss(nn.Module):
    def __init__(self, vocab_size, pad_id, smoothing):
        super().__init__()
        self.pad_id, self.vocab_size = pad_id, vocab_size
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing

    def forward(self, logits, target):
        logits = logits.reshape(-1, self.vocab_size)
        target = target.reshape(-1)
        logp = F.log_softmax(logits, dim=-1)
        with torch.no_grad():
            dist = torch.full_like(logp, self.smoothing / (self.vocab_size - 2))
            dist.scatter_(1, target.unsqueeze(1), self.confidence)
            dist[:, self.pad_id] = 0
            pad_mask = target == self.pad_id
            dist[pad_mask] = 0
        loss = torch.sum(-dist * logp, dim=-1)
        return loss.sum() / (~pad_mask).sum().clamp(min=1)


class NoamSchedule:
    def __init__(self, optimizer, d_model, warmup):
        self.opt, self.d_model, self.warmup, self.step_num = optimizer, d_model, warmup, 0
    def step(self):
        self.step_num += 1
        s = self.step_num
        lr = self.d_model ** -0.5 * min(s ** -0.5, s * self.warmup ** -1.5)
        for g in self.opt.param_groups:
            g["lr"] = lr
        self.opt.step()
        return lr

criterion = LabelSmoothingLoss(cfg.tgt_vocab, cfg.pad_id, cfg.label_smoothing)
optimizer = torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)
scheduler = NoamSchedule(optimizer, cfg.d_model, cfg.warmup_steps)

## 7. Decoding — greedy and beam search

Greedy (fast, used to monitor dev BLEU each epoch) and beam search (ranked by length-normalized
log-prob, used for the final translations).

In [ ]:
@torch.no_grad()
def greedy_decode_batch(model, src, max_len):
    model.eval()
    src = src.to(DEVICE)
    src_mask = model.make_src_mask(src)
    memory = model.encode(src, src_mask)
    B = src.size(0)
    ys = torch.full((B, 1), cfg.bos_id, dtype=torch.long, device=DEVICE)
    done = torch.zeros(B, dtype=torch.bool, device=DEVICE)
    for _ in range(max_len - 1):
        tgt_mask = model.make_tgt_mask(ys)
        out = model.decode(ys, memory, tgt_mask, src_mask)
        nxt = model.generator(out[:, -1]).argmax(-1, keepdim=True)
        ys = torch.cat([ys, nxt], dim=1)
        done = done | (nxt.squeeze(1) == cfg.eos_id)
        if done.all():
            break
    return ys


@torch.no_grad()
def beam_decode_one(model, src_ids, max_len, beam_size, length_penalty):
    model.eval()
    src = src_ids.unsqueeze(0).to(DEVICE)
    src_mask = model.make_src_mask(src)
    memory = model.encode(src, src_mask)

    beams = [(torch.tensor([cfg.bos_id], device=DEVICE), 0.0)]
    finished = []
    for _ in range(max_len - 1):
        pool = []
        for seq, score in beams:
            if seq[-1].item() == cfg.eos_id:
                finished.append((seq, score)); continue
            tgt_mask = model.make_tgt_mask(seq.unsqueeze(0))
            out = model.decode(seq.unsqueeze(0), memory, tgt_mask, src_mask)
            logp = F.log_softmax(model.generator(out[:, -1]), dim=-1).squeeze(0)
            vals, idx = logp.topk(beam_size)
            for v, i in zip(vals, idx):
                pool.append((torch.cat([seq, i.view(1)]), score + v.item()))
        if not pool:
            break
        norm = lambda x: x[1] / (len(x[0]) ** length_penalty)
        pool.sort(key=norm, reverse=True)
        beams = pool[:beam_size]
        if all(s[-1].item() == cfg.eos_id for s, _ in beams):
            finished.extend(beams); break
    pool = finished if finished else beams
    pool.sort(key=lambda x: x[1] / (len(x[0]) ** length_penalty), reverse=True)
    return pool[0][0]


def ids_to_text(ids, sp):
    """Strip special tokens and detokenize back to a normal string."""
    out = []
    for t in (ids.tolist() if torch.is_tensor(ids) else ids):
        if t == cfg.eos_id:
            break
        if t not in (cfg.bos_id, cfg.pad_id):
            out.append(t)
    return sp.decode(out)


def translate_corpus(model, dataset, use_beam=True):
    """Translate an entire dataset, returning a dict {id: english_string}."""
    preds = {}
    loader = DataLoader(dataset, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_infer)
    for ids, src in loader:
        if use_beam:
            for i, one in zip(ids, src):
                real = one[one != cfg.pad_id]
                out = beam_decode_one(model, real, cfg.max_len, cfg.beam_size, cfg.length_penalty)
                preds[i] = ids_to_text(out, sp_tgt)
        else:
            out = greedy_decode_batch(model, src, cfg.max_len)
            for i, row in zip(ids, out):
                preds[i] = ids_to_text(row, sp_tgt)
    return preds

## 8. BLEU helper (matches the grader)

Corpus BLEU with NLTK's default weights, exactly as the assignment specifies. Corpus-level
accumulation avoids the "one short sentence scores 0" artifact of per-sentence BLEU.

In [ ]:
import nltk
from nltk.translate.bleu_score import corpus_bleu

def compute_corpus_bleu(pred_dict, ref_df):
    """Default-weight NLTK corpus BLEU between predictions and references, aligned by id."""
    refs, hyps = [], []
    for _, r in ref_df.iterrows():
        if r["id"] in pred_dict:
            refs.append([str(r["tgt"]).split()])   # list-of-references form
            hyps.append(pred_dict[r["id"]].split())
    if not hyps:
        return 0.0
    return corpus_bleu(refs, hyps)   # default weights (0.25 x4) as specified

## 9. Training

Teacher-forcing loop with gradient clipping. After each epoch: dev loss, dev BLEU (greedy), keep the
best-BLEU checkpoint, early-stop after `patience` stalled epochs.

> On a Colab GPU this is comfortable. On CPU it's slow — shrink `epochs`/`d_model`/`num_layers` for a
> quick check, then scale back up.

In [ ]:
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

@torch.no_grad()
def dev_loss():
    model.eval(); tot, n = 0.0, 0
    for src, tgt in dev_loader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        logits = model(src, tgt[:, :-1])
        loss = criterion(logits, tgt[:, 1:])
        w = (tgt[:, 1:] != cfg.pad_id).sum().item()
        tot += loss.item() * w; n += w
    return tot / max(n, 1)

history = {"train_loss": [], "dev_loss": [], "dev_bleu": []}
best_bleu, best_epoch, bad = -1.0, -1, 0
ckpt_path = f"{cfg.work_dir}/best_model.pt"

for epoch in range(1, cfg.epochs + 1):
    model.train(); running, seen = 0.0, 0
    for src, tgt in tqdm(train_loader, desc=f"Epoch {epoch:02d}", leave=False):
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        optimizer.zero_grad()
        logits = model(src, tgt[:, :-1])
        loss = criterion(logits, tgt[:, 1:])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scheduler.step()
        running += loss.item() * src.size(0); seen += src.size(0)

    tr = running / seen
    dv = dev_loss()
    bleu = compute_corpus_bleu(translate_corpus(model, dev_ds, use_beam=False), dev_df)
    history["train_loss"].append(tr); history["dev_loss"].append(dv); history["dev_bleu"].append(bleu)
    print(f"Epoch {epoch:02d} | train {tr:.3f} | dev {dv:.3f} | dev BLEU {bleu*100:.2f}")

    if bleu > best_bleu:
        best_bleu, best_epoch, bad = bleu, epoch, 0
        torch.save(model.state_dict(), ckpt_path)
    else:
        bad += 1
        if bad >= cfg.patience:
            print(f"Early stopping — no dev-BLEU gain for {cfg.patience} epochs."); break

print(f"\nBest dev BLEU {best_bleu*100:.2f} at epoch {best_epoch}. Restoring best checkpoint.")
model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))

## 10. Save the config for the evaluation notebook

`evaluate.ipynb` rebuilds the exact model from this JSON, then loads `best_model.pt`. This is what
makes the two notebooks reproducible against the private test set.

In [ ]:
config_out = {
    "src_vocab": cfg.src_vocab, "tgt_vocab": cfg.tgt_vocab,
    "pad_id": cfg.pad_id, "unk_id": cfg.unk_id, "bos_id": cfg.bos_id, "eos_id": cfg.eos_id,
    "d_model": cfg.d_model, "num_heads": cfg.num_heads, "num_layers": cfg.num_layers,
    "d_ff": cfg.d_ff, "dropout": cfg.dropout, "max_len": cfg.max_len,
    "beam_size": cfg.beam_size, "length_penalty": cfg.length_penalty,
    "batch_size": cfg.batch_size, "best_epoch": best_epoch,
}
with open(f"{cfg.work_dir}/config.json", "w") as f:
    json.dump(config_out, f, indent=2)
print("Saved:", ", ".join(os.listdir(cfg.work_dir)))

## 11. Training curves

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ep = range(1, len(history["train_loss"]) + 1)
ax[0].plot(ep, history["train_loss"], label="train")
ax[0].plot(ep, history["dev_loss"], label="dev")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(ep, [b*100 for b in history["dev_bleu"]], color="green")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("BLEU"); ax[1].set_title("Dev BLEU")
plt.tight_layout()
plt.savefig(f"{cfg.work_dir}/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Evaluate — BLEU and BERTScore on dev and test

Beam-search decode, then the two metrics. **BERTScore** uses F1 with `rescale_with_baseline=True`
and `lang="en"`, as specified (its first call downloads a scoring model, ~1 min). The provided test
set has references, so we report both dev and test here for the report.

In [ ]:
from bert_score import score as bertscore_score

def evaluate_split(dataset, ref_df, name):
    preds = translate_corpus(model, dataset, use_beam=True)
    bleu = compute_corpus_bleu(preds, ref_df)
    ids  = [i for i in ref_df["id"] if i in preds]
    cands = [preds[i] for i in ids]
    refs  = [str(ref_df.loc[ref_df["id"] == i, "tgt"].iloc[0]) for i in ids]
    _, _, F1 = bertscore_score(cands, refs, lang="en", rescale_with_baseline=True)
    f1 = F1.mean().item()
    print(f"[{name}] BLEU {bleu*100:.2f} | BERTScore F1 {f1:.4f}")
    return preds, bleu, f1

dev_preds,  dev_bleu,  dev_f1  = evaluate_split(dev_ds,  dev_df,  "dev")
test_preds, test_bleu, test_f1 = evaluate_split(test_ds, test_df, "test")

## 13. Efficiency — inference time and parameter count

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

start = time.time()
_ = translate_corpus(model, test_ds, use_beam=True)   # time the full test-set decode
inference_time = time.time() - start
print(f"Test set ({len(test_ds)} sentences) decoded in {inference_time:.2f}s "
      f"({inference_time/len(test_ds)*1000:.1f} ms/sentence, beam={cfg.beam_size})")

## 14. Write `submission.csv`

Columns `Source_id, Sentence_en`, UTF-8, in the original test order.

In [ ]:
order = {sid: k for k, sid in enumerate(test_df["id"].tolist())}
submission = (pd.DataFrame({"Source_id": list(test_preds.keys()),
                            "Sentence_en": list(test_preds.values())})
              .sort_values("Source_id", key=lambda s: s.map(order))
              .reset_index(drop=True))
submission.to_csv("submission.csv", index=False, encoding="utf-8")
print("Wrote submission.csv:", len(submission), "rows")
submission.head()

## 15. Translation examples for the report

In [ ]:
import textwrap
for i, sid in enumerate(list(dev_preds.keys())[:10], 1):
    src = dev_df.loc[dev_df["id"] == sid, "src"].iloc[0]
    ref = dev_df.loc[dev_df["id"] == sid, "tgt"].iloc[0]
    print(f"[{i}] id={sid}")
    print("  SA :", textwrap.shorten(src, 100))
    print("  REF:", textwrap.shorten(str(ref), 100))
    print("  OUT:", textwrap.shorten(dev_preds[sid], 100)); print()

## 16. Results summary and disclosure

In [ ]:
print("="*48)
print("            RESULTS SUMMARY")
print("="*48)
print(f"Dev  BLEU {dev_bleu*100:6.2f} | BERTScore F1 {dev_f1:.4f}")
print(f"Test BLEU {test_bleu*100:6.2f} | BERTScore F1 {test_f1:.4f}")
print(f"Inference time (test): {inference_time:.2f} s")
print(f"Total parameters:      {total_params:,}")
print(f"Best epoch:            {best_epoch}")
print("="*48)
print("""
Pre-trained model disclosure
----------------------------
Translation model : trained from scratch on the provided data only;
                    no pre-trained translation weights used.
Evaluation only   : BERTScore internally loads a pre-trained RoBERTa-large
                    model to compute its metric. It plays no role in
                    generating translations.
""")

---
### Commit these to your public GitHub repo
- `train.ipynb` (this notebook) and `evaluate.ipynb`
- `artifacts/` — `best_model.pt`, `spm_sa.model`, `spm_en.model`, `config.json`, `training_curves.png`
- `submission.csv`
- the report PDF

At evaluation time, open `evaluate.ipynb`, point it at the released private test file, and run it —
it reloads these artifacts and produces the final `submission.csv` and scores.